In [2]:
# !pip install torch
# !pip install transformers
# !pip install datasets
# !pip install trl
# !pip install peft
# !pip install accelerate
# !pip install bitsandbytes
# !pip install scikit-learn
# !pip install numpy
# !pip install wandb
# !pip install tqdm

In [2]:
import torch
import numpy as np
import random
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, TrainerCallback
from datasets import Dataset
import os
import json
import re
import wandb
import pandas as pd

import trl
from trl import GRPOConfig, GRPOTrainer
print(f"trl version: {trl.__version__}")

# Set seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

trl version: 1.4.0


# 1. Model Selection and Loading

In [15]:
# Options for small models
model_options = {
    "qwen-0.5b-instruct": "Qwen/Qwen2.5-0.5B-Instruct",
    "qwen-1.5b-instruct": "Qwen/Qwen2.5-1.5B-Instruct",
    "tinyllama": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
    "gpt2": "gpt2",  # 124M parameters
    "gpt2-medium": "gpt2-medium",  # 355M parameters
    "opt-125m": "facebook/opt-125m",
    "bloom-560m": "bigscience/bloom-560m"
}

# Choose a model
MODEL_CHOICE = "gpt2"
model_name = model_options[MODEL_CHOICE]

print(f"Selected model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the model with optimal settings for limited resources
device_map = "auto"
if torch.cuda.is_available():
    print("Using CUDA")
elif torch.backends.mps.is_available():
    print("CUDA unavailable, using MPS for Mac")
    device_map = {"": "mps"}
else:
    print("CUDA and MPS unavailable, using CPU")
    device_map = {"": "cpu"}

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    torch_dtype=torch.bfloat16,
    device_map=device_map,
    low_cpu_mem_usage=True
)

Selected model: gpt2
Using CUDA


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# 2. Dataset Creation

In [21]:
def generate_math_problem():
    """Generates a random math problem."""
    operations = ["+", "-", "*", "/"]
    op = random.choice(operations)

    if op == "+":
        a = random.randint(1, 100)
        b = random.randint(1, 100)
        answer = a + b
        prompt = f"What is {a} plus {b}?"
    elif op == "-":
        a = random.randint(1, 100)
        b = random.randint(1, min(a, 100))  # To ensure a positive result
        answer = a - b
        prompt = f"What is {a} minus {b}?"
    elif op == "*":
        a = random.randint(1, 20)
        b = random.randint(1, 20)
        answer = a * b
        prompt = f"What is {a} multiplied by {b}?"
    else:  # "/"
        b = random.randint(1, 10)
        a = b * random.randint(1, 10)  # To ensure an integer result
        answer = a // b
        prompt = f"What is {a} divided by {b}?"

    return prompt, answer

# Create training and validation datasets
def create_datasets(train_size=200, val_size=50):
    """Creates training and validation datasets."""
    all_data = []

    # Generate data
    for _ in range(train_size + val_size):
        prompt, answer = generate_math_problem()
        all_data.append({"prompt": prompt, "answer": answer})

    random.shuffle(all_data)

    train_data = all_data[:train_size]
    val_data = all_data[train_size:]

    train_dataset = Dataset.from_list(train_data)
    val_dataset = Dataset.from_list(val_data)

    return train_dataset, val_dataset

In [22]:
def reward_function(prompts, completions, **kwargs):
    """
    Reward function for evaluating model responses.

    Parameters:
    - prompts: list of problems
    - completions: list of model responses
    - kwargs: additional arguments that GRPOTrainer may pass

    Returns:
    - list of rewards for each response
    """
    rewards = []

    ground_truth = kwargs.get('answer', [None] * len(prompts))

    for prompt, completion, gt in zip(prompts, completions, ground_truth):
        reward = 0.0

        # Extract numerical answer
        match = re.search(r'^\s*Answer:\s*(\d+)', completion)

        format_correct = bool(match)

        if format_correct:
            reward += 1.0

            # Check answer correctness if ground_truth is available
            if gt is not None:
                try:
                    provided_answer = int(match.group(1))

                    after_number = completion[match.end():].strip() # Penalty for extra tokens after answer
                    if after_number:
                        extra_tokens = len(after_number.split())

                        reward -= 1.0 * extra_tokens

                    # Differentiated reward based on closeness to correct answer
                    if provided_answer == gt:
                        reward += 2.0  # Full reward for correct answer
                    elif abs(provided_answer - gt) <= 5:
                        reward += 1.0  # Partial reward for close answer
                    elif abs(provided_answer - gt) <= 10:
                        reward += 0.5  # Small reward for not too distant answer
                except:
                    pass

        rewards.append(reward)

    return rewards

In [23]:
def format_prompt(problem):
    return (
        "Solve the arithmetic problem.\n"
        "Output exactly one line in the format: Answer: <integer>\n"
        f"Problem: {problem}\n"
    )

# 3. Supervised Fine-Tuning

In [24]:
def create_datasets_sft(train_size=1000, val_size=100):
    all_data = []

    for _ in range(train_size + val_size):
        prompt, answer = generate_math_problem()
        text = format_prompt(prompt) + f"Answer: {answer}"
        all_data.append({"text": text})

    random.shuffle(all_data)

    train_data = all_data[:train_size]
    val_data = all_data[train_size:]

    return Dataset.from_list(train_data), Dataset.from_list(val_data)

In [25]:
train_dataset, val_dataset = create_datasets_sft(train_size=1000, val_size=100)

print("Examples from the training dataset:")
for i in range(3):
    print(f"Example {i+1}: {train_dataset[i]}")

Examples from the training dataset:
Example 1: {'text': 'Solve the arithmetic problem.\nOutput exactly one line in the format: Answer: <integer>\nProblem: What is 74 plus 77?\nAnswer: 151'}
Example 2: {'text': 'Solve the arithmetic problem.\nOutput exactly one line in the format: Answer: <integer>\nProblem: What is 77 minus 15?\nAnswer: 62'}
Example 3: {'text': 'Solve the arithmetic problem.\nOutput exactly one line in the format: Answer: <integer>\nProblem: What is 40 divided by 10?\nAnswer: 4'}


## 3.1. Model evaluation before SFT

In [26]:
def evaluate_reward(model, tokenizer, test_dataset=None, num_examples=50):
    if test_dataset is None:
        eval_data = []
        for _ in range(num_examples):
            prompt, answer = generate_math_problem()
            eval_data.append({"prompt": prompt, "answer": answer})
        test_dataset = Dataset.from_list(eval_data)

    total_reward = 0.0
    examples_with_rewards = []

    print("Generating responses and calculating rewards...")

    model.eval()

    for example in tqdm(test_dataset):
        problem = example["prompt"]
        expected = example["answer"]

        formatted_prompt = format_prompt(problem)

        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=8,
                min_new_tokens=1,
                pad_token_id=tokenizer.eos_token_id
            )

        generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        completion = tokenizer.decode(generated_tokens, skip_special_tokens=True)

        reward = reward_function([formatted_prompt], [completion], answer=[expected])[0]
        total_reward += reward

        examples_with_rewards.append({
            "problem": problem,
            "expected": expected,
            "response": completion,
            "reward": reward
        })

    avg_reward = total_reward / len(test_dataset)
    print(f"Average reward: {avg_reward:.4f}")

    return avg_reward, examples_with_rewards

In [27]:
# Test dataset for evaluation
eval_size = 50
test_data = []
for _ in range(eval_size):
    prompt, answer = generate_math_problem()
    test_data.append({"prompt": prompt, "answer": answer})
test_dataset = Dataset.from_list(test_data)

# Evaluate the original model
original_avg_reward, original_examples = evaluate_reward(model, tokenizer, test_dataset)

Generating responses and calculating rewards...


  0%|          | 0/50 [00:00<?, ?it/s]

Average reward: 0.0000


## 3.2. Starting SFT

In [28]:
training_args = trl.SFTConfig(
    output_dir="./results",
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch",
    max_length=64,
    eval_strategy="epoch",
    run_name=f"sft-{MODEL_CHOICE}",
)

trainer = trl.SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    eval_dataset=val_dataset,
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Epoch,Training Loss,Validation Loss
1,0.409547,0.379704


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=125, training_loss=0.49574487495422365, metrics={'train_runtime': 39.2675, 'train_samples_per_second': 25.466, 'train_steps_per_second': 3.183, 'total_flos': 17833181184000.0, 'train_loss': 0.49574487495422365})

## 3.3. Model validation after SFT

In [29]:
# just fine-tuned model
model = AutoModelForCausalLM.from_pretrained(
    './results/checkpoint-125',
    torch_dtype=torch.bfloat16,
    device_map=device_map,
    low_cpu_mem_usage=True
)

sft_avg_reward, sft_examples = evaluate_reward(model, tokenizer, test_dataset)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Generating responses and calculating rewards...


  0%|          | 0/50 [00:00<?, ?it/s]

Average reward: 1.4800


# 4. GRPO Training Preparation

In [30]:
def create_datasets(train_size=200, val_size=50):
    """Creates training and validation datasets."""
    all_data = []

    for _ in range(train_size + val_size):
        prompt, answer = generate_math_problem()
        all_data.append({"prompt": prompt, "answer": answer})

    random.shuffle(all_data)

    train_data = all_data[:train_size]
    val_data = all_data[train_size:]

    train_dataset = Dataset.from_list(train_data)
    val_dataset = Dataset.from_list(val_data)

    return train_dataset, val_dataset

In [31]:
train_dataset, val_dataset = create_datasets(train_size=100, val_size=100)

train_dataset = train_dataset.map(lambda x: {
    "prompt": format_prompt(x["prompt"]),
    "answer": x["answer"]
})

val_dataset = val_dataset.map(lambda x: {
    "prompt": format_prompt(x["prompt"]),
    "answer": x["answer"]
})

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [32]:
for i in range(3):
    prompt = train_dataset[i]["prompt"]
    expected = train_dataset[i]["answer"]

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=8,
        min_new_tokens=1,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    print(f"\nProblem: {prompt}")
    print(f"Expected answer: {expected}")
    print(f"Model completion: {response}")


Problem: Solve the arithmetic problem.
Output exactly one line in the format: Answer: <integer>
Problem: What is 81 divided by 9?

Expected answer: 9
Model completion: Answer: 9

Problem: Solve the arithmetic problem.
Output exactly one line in the format: Answer: <integer>
Problem: What is 1 multiplied by 4?

Expected answer: 4
Model completion: Answer: 4

Problem: Solve the arithmetic problem.
Output exactly one line in the format: Answer: <integer>
Problem: What is 14 multiplied by 16?

Expected answer: 224
Model completion: Answer: 168


In [33]:
grpo_config = GRPOConfig(
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,  # Increased gradient accumulation steps
    # beta=0.1,  # Target KL divergence
    seed=SEED,
    scale_rewards=True,  # Reward scaling
    output_dir="./grpo_checkpoint",  # Directory for checkpoint saving
    logging_steps=10,  # Log every 10 steps
    save_strategy="epoch",  # Save model at the end of each epoch
    eval_strategy="epoch",  # Evaluate model at the end of each epoch
    num_train_epochs=3,
    max_completion_length=4,
    num_generations=4,
    report_to=["wandb"],  # Enable wandb reports
    log_completions=True,  # Enable text example logging
    # wandb_log_unique_prompts=True  # Log only unique prompts to save space
)

# GRPO trainer
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=grpo_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    reward_funcs=[reward_function],
)

In [34]:
def prepare_dataset_for_grpo(dataset):
    problems = [item["prompt"] for item in dataset]
    expected_answers = [item["answer"] for item in dataset]

    initial_responses = []
    print("Getting initial model responses...")

    model.eval()

    with torch.no_grad():
        for problem in tqdm(problems[:50]):
            inputs = tokenizer(problem, return_tensors="pt").to(model.device)
            outputs = model.generate(
                **inputs,
                max_new_tokens=8,
                min_new_tokens=1,
                pad_token_id=tokenizer.eos_token_id
            )

            generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
            response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
            initial_responses.append(response)

    initial_rewards = reward_function(
        problems[:50],
        initial_responses,
        answer=expected_answers[:50]
    )

    avg_initial_reward = sum(initial_rewards) / len(initial_rewards)
    print(f"Average initial reward on sample: {avg_initial_reward:.2f}")

# 5. GRPO Training

In [35]:
def evaluate_model_for_wandb(model, tokenizer, test_problems=None, epoch=0):
    """
    Evaluates the model on test examples and logs results to wandb.

    Parameters:
    - model: trained model
    - tokenizer: tokenizer
    - test_problems: list of test problems (if None, predefined ones are used)
    - epoch: epoch number
    """
    if test_problems is None:
        test_problems = [
            "What is 15 plus 27?",
            "What is 42 minus 18?",
            "What is 7 multiplied by 8?",
            "What is 45 divided by 9?"
        ]

    results = []

    for problem in test_problems:
        formatted_prompt = format_prompt(problem)

        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(
            inputs["input_ids"],
            max_new_tokens=4,
            min_new_tokens=1,
            pad_token_id=tokenizer.eos_token_id
        )
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        match = re.search(r'Answer:\s*(\d+)', response)
        answer = match.group(1) if match else "Invalid format"

        results.append({
            "Problem": problem,
            "Model Response": response,
            "Numerical Answer": answer
        })

    if wandb.run is not None:
        wandb.log({
            "model_samples/epoch": epoch,
            "model_samples/examples": wandb.Table(
                dataframe=pd.DataFrame(results)
            )
        })

    return results


class WandbEvalCallback(TrainerCallback):
    """Callback for logging generation results to wandb after each epoch"""

    def __init__(self, model, tokenizer, test_problems=None):
        self.model = model
        self.tokenizer = tokenizer
        self.test_problems = test_problems

    def on_epoch_end(self, args, state, control, **kwargs):
        evaluate_model_for_wandb(
            self.model,
            self.tokenizer,
            self.test_problems,
            state.epoch
        )
        return control

if "wandb" in grpo_config.report_to:
    test_problems = [
        "What is 15 plus 27?",
        "What is 42 minus 18?",
        "What is 7 multiplied by 8?",
        "What is 45 divided by 9?",
        "What is 33 plus 44?",
        "What is 99 minus 34?",
        "What is 12 multiplied by 5?",
        "What is 72 divided by 8?"
    ]
    trainer.add_callback(WandbEvalCallback(model, tokenizer, test_problems))

In [36]:
train_subset = train_dataset.select(range(min(100, len(train_dataset))))
prepare_dataset_for_grpo(train_subset)

print("\nStarting GRPO training...")

trainer.train()

Getting initial model responses...


  0%|          | 0/50 [00:00<?, ?it/s]

Average initial reward on sample: 1.48

Starting GRPO training...


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


...

╭──────────────────────────────────────────────── Step 75 ─────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                                   ┃ Completion  ┃ reward_function ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ Solve the arithmetic problem.                            │ Answer: 44  │            1.00 │     -0.50 │ │
│ │ Output exactly one line in the format: Answer: <integer> │             │                 │           │ │
│ │ Problem: What is 31 plus 73?                             │             │                 │           │ │
│ │                                                          │             │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼─────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 98  │            1.50 │      1.50 │ │
│ │ Output exactly one line in the format: Answer: <integer> │             │                 │           │ │
│ │ Problem: What is 31 plus 73?                             │             │                 │           │ │
│ │                                                          │             │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼─────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 49  │            1.00 │     -0.50 │ │
│ │ Output exactly one line in the format: Answer: <integer> │             │                 │           │ │
│ │ Problem: What is 31 plus 73?                             │             │                 │           │ │
│ │                                                          │             │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼─────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 77  │            1.00 │     -0.50 │ │
│ │ Output exactly one line in the format: Answer: <integer> │             │                 │           │ │
│ │ Problem: What is 31 plus 73?                             │             │                 │           │ │
│ │                                                          │             │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼─────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 85  │            1.00 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │             │                 │           │ │
│ │ Problem: What is 92 minus 90?                            │             │                 │           │ │
│ │                                                          │             │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼─────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 17  │            1.00 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │             │                 │           │ │
│ │ Problem: What is 92 minus 90?                            │             │                 │           │ │
│ │                                                          │             │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼─────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 144 │            1.00 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │             │                 │           │ │
│ │ Problem: What is 92 minus 90?                            │             │

wandb: WARNING URL not available in offline run


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

╭──────────────────────────────────────────────── Step 75 ─────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                                   ┃ Completion  ┃ reward_function ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ Solve the arithmetic problem.                            │ Answer: 44  │            1.00 │     -0.50 │ │
│ │ Output exactly one line in the format: Answer: <integer> │             │                 │           │ │
│ │ Problem: What is 31 plus 73?                             │             │                 │           │ │
│ │                                                          │             │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼─────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 98  │            1.50 │      1.50 │ │
│ │ Output exactly one line in the format: Answer: <integer> │             │                 │           │ │
│ │ Problem: What is 31 plus 73?                             │             │                 │           │ │
│ │                                                          │             │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼─────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 49  │            1.00 │     -0.50 │ │
│ │ Output exactly one line in the format: Answer: <integer> │             │                 │           │ │
│ │ Problem: What is 31 plus 73?                             │             │                 │           │ │
│ │                                                          │             │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼─────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 77  │            1.00 │     -0.50 │ │
│ │ Output exactly one line in the format: Answer: <integer> │             │                 │           │ │
│ │ Problem: What is 31 plus 73?                             │             │                 │           │ │
│ │                                                          │             │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼─────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 85  │            1.00 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │             │                 │           │ │
│ │ Problem: What is 92 minus 90?                            │             │                 │           │ │
│ │                                                          │             │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼─────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 17  │            1.00 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │             │                 │           │ │
│ │ Problem: What is 92 minus 90?                            │             │                 │           │ │
│ │                                                          │             │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼─────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 144 │            1.00 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │             │                 │           │ │
│ │ Problem: What is 92 minus 90?                            │             │

TrainOutput(global_step=75, training_loss=-1.7384688059488933e-09, metrics={'train_runtime': 99.2561, 'train_samples_per_second': 3.022, 'train_steps_per_second': 0.756, 'total_flos': 0.0, 'train_loss': -1.7384688059488933e-09})

# 6. Final Comparison Before/After

In [37]:
def generate_completion(model, tokenizer, prompt, max_new_tokens=8):
    model.eval()

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=1,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    completion = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return completion


print("\n===== Final comparison: SFT model vs SFT + GRPO model =====")

sft_model_for_comparison = AutoModelForCausalLM.from_pretrained(
    "./results/checkpoint-125",
    torch_dtype=torch.bfloat16,
    device_map=device_map,
    low_cpu_mem_usage=True
)

test_problems = [
    "What is 15 plus 27?",
    "What is 42 minus 18?",
    "What is 7 multiplied by 8?",
    "What is 45 divided by 9?"
]

print("\nComparison of responses on new examples:")

for problem in test_problems:
    prompt = format_prompt(problem)

    sft_completion = generate_completion(
        sft_model_for_comparison,
        tokenizer,
        prompt,
        max_new_tokens=8
    )

    grpo_completion = generate_completion(
        trainer.model,
        tokenizer,
        prompt,
        max_new_tokens=8
    )

    print("=" * 60)
    print(f"Problem: {problem}")
    print(f"SFT model completion:      {repr(sft_completion)}")
    print(f"SFT + GRPO completion:     {repr(grpo_completion)}")
    print("=" * 60)


print("\n===== Evaluating SFT + GRPO model =====")

grpo_avg_reward, grpo_examples = evaluate_reward(
    trainer.model,
    tokenizer,
    test_dataset
)

print("\n===== Reward comparison =====")
print(f"Average reward BEFORE SFT:      {original_avg_reward:.4f}")
print(f"Average reward AFTER SFT:       {sft_avg_reward:.4f}")
print(f"Average reward AFTER SFT+GRPO:  {grpo_avg_reward:.4f}")

print(f"SFT improvement:       {sft_avg_reward - original_avg_reward:.4f}")
print(f"GRPO improvement over SFT: {grpo_avg_reward - sft_avg_reward:.4f}")


print("\n===== Examples comparison =====")

for i, (orig, sft, grpo) in enumerate(
    zip(original_examples[:5], sft_examples[:5], grpo_examples[:5])
):
    print(f"\nExample {i+1}:")
    print(f"Problem: {orig['problem']}")
    print(f"Expected: {orig['expected']}")
    print(f"Original model response: {orig['response']} (reward: {orig['reward']:.2f})")
    print(f"SFT model response:      {sft['response']} (reward: {sft['reward']:.2f})")
    print(f"SFT+GRPO response:       {grpo['response']} (reward: {grpo['reward']:.2f})")


if "wandb" in grpo_config.report_to and wandb.run is not None:
    comparison_data = []

    for orig, sft, grpo in zip(original_examples, sft_examples, grpo_examples):
        comparison_data.append({
            "Problem": orig["problem"],
            "Expected Answer": orig["expected"],
            "Original Response": orig["response"],
            "Original Reward": orig["reward"],
            "SFT Response": sft["response"],
            "SFT Reward": sft["reward"],
            "SFT+GRPO Response": grpo["response"],
            "SFT+GRPO Reward": grpo["reward"],
            "SFT Improvement": sft["reward"] - orig["reward"],
            "GRPO Improvement over SFT": grpo["reward"] - sft["reward"],
        })

    wandb.log({
        "final_comparison": wandb.Table(dataframe=pd.DataFrame(comparison_data)),
        "original_avg_reward": original_avg_reward,
        "sft_avg_reward": sft_avg_reward,
        "grpo_avg_reward": grpo_avg_reward,
        "sft_reward_improvement": sft_avg_reward - original_avg_reward,
        "grpo_reward_improvement_over_sft": grpo_avg_reward - sft_avg_reward,
    })

print("\nSeminar completed! The model has been trained with SFT and then GRPO.")


===== Final comparison: SFT model vs SFT + GRPO model =====


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Comparison of responses on new examples:
Problem: What is 15 plus 27?
SFT model completion:      'Answer: 100'
SFT + GRPO completion:     'Answer: 100'
Problem: What is 42 minus 18?
SFT model completion:      'Answer: 1'
SFT + GRPO completion:     'Answer: 1'
Problem: What is 7 multiplied by 8?
SFT model completion:      'Answer: 168'
SFT + GRPO completion:     'Answer: 32'
Problem: What is 45 divided by 9?
SFT model completion:      'Answer: 2'
SFT + GRPO completion:     'Answer: 3'

===== Evaluating SFT + GRPO model =====
Generating responses and calculating rewards...


  0%|          | 0/50 [00:00<?, ?it/s]

Average reward: 1.4300

===== Reward comparison =====
Average reward BEFORE SFT:      0.0000
Average reward AFTER SFT:       1.4800
Average reward AFTER SFT+GRPO:  1.4300
SFT improvement:       1.4800
GRPO improvement over SFT: -0.0500

===== Examples comparison =====

Example 1:
Problem: What is 58 minus 5?
Expected: 53
Original model response: Answer: <integer>
Solution: (reward: 0.00)
SFT model response:      Answer: 2 (reward: 1.00)
SFT+GRPO response:       Answer: 1 (reward: 1.00)

Example 2:
Problem: What is 17 multiplied by 13?
Expected: 221
Original model response: Answer: <integer>
Solution: (reward: 0.00)
SFT model response:      Answer: 168 (reward: 1.00)
SFT+GRPO response:       Answer: 160 (reward: 1.00)

Example 3:
Problem: What is 50 divided by 5?
Expected: 10
Original model response: Solution: The answer is 50 divided by (reward: 0.00)
SFT model response:      Answer: 5 (reward: 2.00)
SFT+GRPO response:       Answer: 5 (reward: 2.00)

Example 4:
Problem: What is 56 divi